In [ ]:
#!/usr/bin/env python3
"""
Precision Dissociation in Narcissistic Personality Disorder:
A Hierarchical Predictive Coding Simulation

Hypothesis: NPD is characterised by opposing precision assignments across
two hierarchical empathy inference streams:
  (1) Cognitive stream  (ToM / mentalising): HIGH precision π_cog  → intact accuracy
  (2) Affective stream  (resonance / affect): LOW  precision π_aff  → impaired concern

Additionally, the self-model carries an inflated, high-precision prior.
Under sustained ego-threat the self-model precision erodes → narcissistic collapse.

Author : M. Fronzi (2026)
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from scipy import stats
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────
# AESTHETICS
# ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'axes.linewidth': 0.8,
})

C_HC  = '#2166ac'   # blue  – healthy controls
C_NPD = '#d6604d'   # red   – NPD
C_INJ = '#fdae61'   # amber – injury block
ALPHA_SHADE = 0.18

# ─────────────────────────────────────────────────────────────────────
# MODEL PARAMETERS
# ─────────────────────────────────────────────────────────────────────
N_AGENTS = 50
T        = 300          # total trials per agent

INJURY_START = 150      # narcissistic-injury block onset
INJURY_LEN   = 30       # duration (trials)

# Precision parameters  (π = inverse variance; larger → tighter prior / more weight)
PARAMS = {
    'HC' : dict(pi_cog=3.0, pi_aff=3.0, pi_self=2.0, mu_self_0=0.0),
    'NPD': dict(pi_cog=5.0, pi_aff=0.6, pi_self=9.0, mu_self_0=1.5),
}

# Observation noise
SIGMA_OBS_COG = 0.50
SIGMA_OBS_AFF = 0.50

# Social-signal prior precision (uninformative)
PI_S_PRIOR = 1.0
PI_A_PRIOR = 1.0

# Self-referential feedback channel precision
PI_FEEDBACK = 1.0

# Threat accumulation / precision decay
CUM_THREAT_INCREMENT = 1.0    # added per threatening trial
CUM_THREAT_DECAY     = 0.50   # subtracted per non-threatening trial
THREAT_DECAY_RATE    = 0.05   # γ  in  π_self(t) = π_self_0 · exp(−γ·C(t))

# ─────────────────────────────────────────────────────────────────────
# STIMULUS GENERATION
# ─────────────────────────────────────────────────────────────────────
_STIM_NORMAL = [0.40, 0.30, 0.30]   # enhance / neutral / threaten
_STIM_INJURY = [0.00, 0.10, 0.90]

def make_stim_sequence(T, inj_start, inj_len, rng):
    out = np.empty(T, dtype=int)
    for t in range(T):
        p = _STIM_INJURY if inj_start <= t < inj_start + inj_len else _STIM_NORMAL
        out[t] = rng.choice(3, p=p)
    return out

def stim_values(st, rng):
    """True (s, a) for other agent and self-referential feedback f."""
    if   st == 0: mu_s, mu_a, mu_f =  0.70,  0.60,  1.00
    elif st == 1: mu_s, mu_a, mu_f =  0.00,  0.00,  0.00
    else:         mu_s, mu_a, mu_f = -0.70, -0.60, -1.00
    s = rng.normal(mu_s, 0.25)
    a = rng.normal(mu_a, 0.25)
    f = rng.normal(mu_f, 0.30)
    return s, a, f

# ─────────────────────────────────────────────────────────────────────
# SINGLE-AGENT SIMULATION
# ─────────────────────────────────────────────────────────────────────
def simulate_agent(params, T, inj_start, inj_len, rng):
    pi_cog    = params['pi_cog']
    pi_aff    = params['pi_aff']
    pi_self_0 = params['pi_self']
    S         = params['mu_self_0']   # current self-esteem estimate

    stims = make_stim_sequence(T, inj_start, inj_len, rng)

    acc     = np.zeros(T)     # empathic accuracy   (cognitive stream)
    con     = np.zeros(T)     # empathic concern    (affective stream)
    SE      = np.zeros(T + 1) # self-esteem trajectory
    PI_S    = np.zeros(T + 1) # dynamic self-precision trajectory
    SE[0]   = S
    PI_S[0] = pi_self_0

    cum_threat = 0.0

    for t in range(T):
        st        = stims[t]
        s_true, a_true, feedback = stim_values(st, rng)

        # ── COGNITIVE STREAM  (Bayesian posterior mean, prior = 0)
        o_cog    = s_true + rng.normal(0, SIGMA_OBS_COG)
        K_cog    = pi_cog / (pi_cog + PI_S_PRIOR)
        mu_s_hat = K_cog * o_cog

        # ── AFFECTIVE STREAM
        o_aff    = a_true + rng.normal(0, SIGMA_OBS_AFF)
        K_aff    = pi_aff / (pi_aff + PI_A_PRIOR)
        mu_a_hat = K_aff * o_aff

        # ── EMPATHY METRICS
        # Accuracy: normalised distance from true other-mental-state
        acc[t] = 1.0 - np.clip(np.abs(mu_s_hat - s_true) / 1.5, 0.0, 1.0)

        # Concern: precision-gated affective resonance magnitude
        # K_aff encodes how strongly affective signal is integrated;
        # total concern = weighted absolute posterior (high π_aff → large K_aff → large concern)
        con[t] = K_aff * np.abs(mu_a_hat)

        # ── SELF-MODEL DYNAMICS
        # Cumulative threat load  C(t)
        if st == 2:
            cum_threat += CUM_THREAT_INCREMENT
        else:
            cum_threat = max(0.0, cum_threat - CUM_THREAT_DECAY)

        # Dynamic self-precision (erodes exponentially under threat load)
        pi_self = max(pi_self_0 * np.exp(-THREAT_DECAY_RATE * cum_threat), 0.05)

        # Precision-weighted Bayesian update of self-esteem
        # S_post = [π_self·S + π_fb·f] / (π_self + π_fb)
        S = (pi_self * S + PI_FEEDBACK * feedback) / (pi_self + PI_FEEDBACK)

        SE[t + 1]   = S
        PI_S[t + 1] = pi_self

    return dict(acc=acc, con=con, SE=SE, PI_S=PI_S, stims=stims)

# ─────────────────────────────────────────────────────────────────────
# ENSEMBLE SIMULATION
# ─────────────────────────────────────────────────────────────────────
master_rng = np.random.default_rng(42)

def run_ensemble(group_key):
    p = PARAMS[group_key]
    return [simulate_agent(p, T, INJURY_START, INJURY_LEN,
                           np.random.default_rng(master_rng.integers(int(1e7))))
            for _ in range(N_AGENTS)]

res = {'HC': run_ensemble('HC'), 'NPD': run_ensemble('NPD')}

def agg(group, key):
    data = np.array([r[key] for r in res[group]])
    return np.mean(data, axis=0), stats.sem(data, axis=0), data

# ─────────────────────────────────────────────────────────────────────
# STATISTICAL TESTS (per-agent scalar summaries)
# ─────────────────────────────────────────────────────────────────────
def scalar_summary(group, key, window=None):
    """Per-agent mean of a metric (optionally over a time window)."""
    data = np.array([r[key] for r in res[group]])
    if window is not None:
        data = data[:, window[0]:window[1]]
    return np.mean(data, axis=1)

# Mean accuracy & concern (whole session)
hc_acc_scalar  = scalar_summary('HC',  'acc')
npd_acc_scalar = scalar_summary('NPD', 'acc')
hc_con_scalar  = scalar_summary('HC',  'con')
npd_con_scalar = scalar_summary('NPD', 'con')

# Self-esteem variability (whole session, T+1 samples)
hc_se_var  = np.array([np.std(r['SE']) for r in res['HC']])
npd_se_var = np.array([np.std(r['SE']) for r in res['NPD']])

# Minimum self-esteem during injury block
hc_se_min  = np.array([np.min(r['SE'][INJURY_START:INJURY_START+INJURY_LEN+1]) for r in res['HC']])
npd_se_min = np.array([np.min(r['SE'][INJURY_START:INJURY_START+INJURY_LEN+1]) for r in res['NPD']])

# Mann-Whitney U tests
def mwu(a, b, label):
    U, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    d = (np.mean(a) - np.mean(b)) / np.sqrt((np.std(a)**2 + np.std(b)**2) / 2)
    print(f"  {label:40s}  HC={np.mean(a):.3f}±{np.std(a):.3f}  "
          f"NPD={np.mean(b):.3f}±{np.std(b):.3f}  U={U:.0f}  p={p:.3e}  d={d:.3f}")
    return p, d

print("\n──── Statistical comparisons ────────────────────────────────────")
mwu(hc_acc_scalar,  npd_acc_scalar,  'Empathic accuracy  (HC vs NPD)')
mwu(hc_con_scalar,  npd_con_scalar,  'Empathic concern   (HC vs NPD)')
mwu(hc_se_var,      npd_se_var,      'Self-esteem std    (HC vs NPD)')
mwu(hc_se_min,      npd_se_min,      'Min SE during injury (HC vs NPD)')
print("─────────────────────────────────────────────────────────────────\n")

# ─────────────────────────────────────────────────────────────────────
# ═══════════════════  FIGURE 1 – MODEL SCHEMATIC  ═══════════════════
# ─────────────────────────────────────────────────────────────────────
fig1, ax = plt.subplots(figsize=(9, 5.5))
ax.set_xlim(0, 10); ax.set_ylim(0, 7)
ax.axis('off')

def box(ax, x, y, w, h, color, label, sublabel='', fontsize=10):
    rect = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.1',
        facecolor=color, edgecolor='#333333', linewidth=1.2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x, y + (0.12 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize, fontweight='bold', color='white')
    if sublabel:
        ax.text(x, y - 0.28, sublabel,
                ha='center', va='center', fontsize=8.5, color='white', style='italic')

def arrow(ax, x0, y0, x1, y1, label='', color='#444444'):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    if label:
        mx, my = (x0+x1)/2, (y0+y1)/2
        ax.text(mx + 0.12, my, label, fontsize=8, color=color, va='center')

# Social world
box(ax, 2.0, 5.8, 2.6, 0.9, '#555577', 'Social Environment',
    '(facial / vocal / behavioural cues)', fontsize=9)

# Sensory layer
box(ax, 1.1, 4.2, 1.6, 0.85, '#3a86ff',
    'Obs. cog.', r'$o_{cog}$  (ToM cue)', fontsize=9)
box(ax, 3.1, 4.2, 1.6, 0.85, '#ff6b6b',
    'Obs. aff.', r'$o_{aff}$  (affect cue)', fontsize=9)

# Inference streams
box(ax, 1.1, 2.6, 1.6, 0.85, '#2166ac',
    'Cognitive stream', r'$\pi_{cog}$ – intact', fontsize=9)
box(ax, 3.1, 2.6, 1.6, 0.85, '#d6604d',
    'Affective stream', r'$\pi_{aff}$ – impaired (NPD)', fontsize=9)

# Self-model
box(ax, 6.5, 4.2, 2.2, 1.0, '#1a9850',
    'Self-Model', r'$S(t)$, precision $\Pi_{self}(t)$', fontsize=9)

# Threat accumulation
box(ax, 6.5, 2.5, 2.2, 0.9, '#8e6bbf',
    'Threat Accumulator', r'$C(t)$  →  $\Pi_{self}$ decay', fontsize=9)

# Outputs
box(ax, 1.1, 1.1, 1.6, 0.75, '#4393c3', 'Empathic Accuracy', fontsize=9)
box(ax, 3.1, 1.1, 1.6, 0.75, '#d73027', 'Empathic Concern', fontsize=9)
box(ax, 6.5, 1.1, 2.2, 0.75, '#1a9850', 'Self-Esteem S(t)', fontsize=9)

# Arrows: environment → observations
arrow(ax, 1.3, 5.35, 1.1, 4.65)
arrow(ax, 2.7, 5.35, 3.1, 4.65)

# Observations → inference
arrow(ax, 1.1, 3.77, 1.1, 3.03, r'$K_{cog}=\frac{\pi_{cog}}{\pi_{cog}+\pi_s^0}$', '#2166ac')
arrow(ax, 3.1, 3.77, 3.1, 3.03, r'$K_{aff}=\frac{\pi_{aff}}{\pi_{aff}+\pi_a^0}$', '#d6604d')

# Inference → output
arrow(ax, 1.1, 2.17, 1.1, 1.48)
arrow(ax, 3.1, 2.17, 3.1, 1.48)

# Environment → self-model feedback
arrow(ax, 3.5, 5.35, 5.8, 4.5)
ax.text(4.5, 5.15, r'$f_t$ (social feedback)', fontsize=8.5, color='#555555', ha='center')

# Self-model → threat accumulator
arrow(ax, 6.5, 3.7, 6.5, 2.97, r'$\varepsilon_{self}=f_t-S$', '#8e6bbf')

# Threat → pi_self decay (feedback loop)
ax.annotate('', xy=(5.4, 4.2), xytext=(5.4, 2.5),
            arrowprops=dict(arrowstyle='->', color='#8e6bbf', lw=1.5,
                            connectionstyle='arc3,rad=-0.3'))
ax.text(4.6, 3.35, r'$\Pi_{self}(t)=\pi_0\,e^{-\gamma C(t)}$',
        fontsize=8.5, color='#8e6bbf', ha='center')

# Self → output
arrow(ax, 6.5, 1.95, 6.5, 1.48)

# NPD vs HC label
ax.text(0.1, 0.35, 'NPD:  $\\pi_{cog}\\gg\\pi_{aff}$,  $\\pi_{self}$ very high (rigid) but collapses under threat',
        fontsize=9, color=C_NPD, style='italic')
ax.text(0.1, 0.05, 'HC:    $\\pi_{cog}\\approx\\pi_{aff}$,  $\\pi_{self}$ moderate (flexible)',
        fontsize=9, color=C_HC, style='italic')

ax.set_title('Hierarchical Predictive Coding Model of Empathy and Self-Esteem in NPD',
             fontsize=12, fontweight='bold', pad=10)
fig1.tight_layout()
fig1.savefig('./fig1_model_schematic.pdf', bbox_inches='tight')
fig1.savefig('./fig1_model_schematic.png', bbox_inches='tight', dpi=150)
print("Fig 1 saved.")

# ─────────────────────────────────────────────────────────────────────
# ═══════════  FIGURE 2 – EMPATHY DISSOCIATION  ═══════════════════════
# ─────────────────────────────────────────────────────────────────────
tt = np.arange(T)
sm = 12   # Gaussian smoothing kernel

fig2, axes = plt.subplots(2, 2, figsize=(11, 7.5),
                           gridspec_kw={'hspace': 0.40, 'wspace': 0.35})

hc_acc_m,  hc_acc_se,  hc_acc_all  = agg('HC',  'acc')
npd_acc_m, npd_acc_se, npd_acc_all = agg('NPD', 'acc')
hc_con_m,  hc_con_se,  hc_con_all  = agg('HC',  'con')
npd_con_m, npd_con_se, npd_con_all = agg('NPD', 'con')

# ── 2A: Accuracy time series
ax = axes[0, 0]
ax.fill_between(tt,
    gaussian_filter1d(hc_acc_m - hc_acc_se, sm),
    gaussian_filter1d(hc_acc_m + hc_acc_se, sm),
    alpha=ALPHA_SHADE, color=C_HC)
ax.fill_between(tt,
    gaussian_filter1d(npd_acc_m - npd_acc_se, sm),
    gaussian_filter1d(npd_acc_m + npd_acc_se, sm),
    alpha=ALPHA_SHADE, color=C_NPD)
ax.plot(tt, gaussian_filter1d(hc_acc_m,  sm), color=C_HC,  lw=2.0, label='HC')
ax.plot(tt, gaussian_filter1d(npd_acc_m, sm), color=C_NPD, lw=2.0, label='NPD')
ax.axvspan(INJURY_START, INJURY_START + INJURY_LEN, color=C_INJ, alpha=0.25,
           label='Injury block')
ax.set_xlabel('Trial'); ax.set_ylabel('Empathic Accuracy')
ax.set_title('(A)  Cognitive stream – Empathic Accuracy\n'
             r'[$1 - |\hat{\mu}_s - s_{true}|/1.5$]')
ax.legend(fontsize=9); ax.set_ylim(0.4, 1.0)

# ── 2B: Concern time series
ax = axes[0, 1]
ax.fill_between(tt,
    gaussian_filter1d(hc_con_m - hc_con_se, sm),
    gaussian_filter1d(hc_con_m + hc_con_se, sm),
    alpha=ALPHA_SHADE, color=C_HC)
ax.fill_between(tt,
    gaussian_filter1d(npd_con_m - npd_con_se, sm),
    gaussian_filter1d(npd_con_m + npd_con_se, sm),
    alpha=ALPHA_SHADE, color=C_NPD)
ax.plot(tt, gaussian_filter1d(hc_con_m,  sm), color=C_HC,  lw=2.0, label='HC')
ax.plot(tt, gaussian_filter1d(npd_con_m, sm), color=C_NPD, lw=2.0, label='NPD')
ax.axvspan(INJURY_START, INJURY_START + INJURY_LEN, color=C_INJ, alpha=0.25)
ax.set_xlabel('Trial'); ax.set_ylabel('Empathic Concern')
ax.set_title('(B)  Affective stream – Empathic Concern\n'
             r'[$K_{aff}\cdot|\hat{\mu}_a|$]')
ax.legend(fontsize=9)

# ── 2C: Per-agent scatter  (accuracy vs. concern)
ax = axes[1, 0]
for xd, yd, col, lab in [(hc_acc_scalar,  hc_con_scalar,  C_HC,  'HC'),
                          (npd_acc_scalar, npd_con_scalar, C_NPD, 'NPD')]:
    ax.scatter(xd, yd, c=col, alpha=0.55, s=28, edgecolors='none', label=lab)
    cx, cy = np.mean(xd), np.mean(yd)
    ax.scatter(cx, cy, c=col, s=120, marker='D', edgecolors='white', linewidths=1.2, zorder=5)
ax.set_xlabel('Mean Empathic Accuracy (Cognitive)'); ax.set_ylabel('Mean Empathic Concern (Affective)')
ax.set_title('(C)  Per-agent scatter: Accuracy vs. Concern')
ax.legend(fontsize=9)
# Add quadrant annotation
ax.axvline(np.mean(hc_acc_scalar), color='grey', lw=0.7, ls='--', alpha=0.5)
ax.axhline(np.mean(hc_con_scalar), color='grey', lw=0.7, ls='--', alpha=0.5)

# ── 2D: Group bar chart with individual dots
ax = axes[1, 1]
metrics  = ['Empathic\nAccuracy', 'Empathic\nConcern']
hc_vals  = [np.mean(hc_acc_scalar),  np.mean(hc_con_scalar)]
npd_vals = [np.mean(npd_acc_scalar), np.mean(npd_con_scalar)]
hc_errs  = [stats.sem(hc_acc_scalar),  stats.sem(hc_con_scalar)]
npd_errs = [stats.sem(npd_acc_scalar), stats.sem(npd_con_scalar)]

x = np.array([0.0, 1.0])
w = 0.28
for xi, hv, nv, he, ne, hd, nd in zip(
        x, hc_vals, npd_vals, hc_errs, npd_errs,
        [hc_acc_scalar, hc_con_scalar],
        [npd_acc_scalar, npd_con_scalar]):
    ax.bar(xi - w/2, hv, width=w, color=C_HC, alpha=0.75,
           yerr=he, capsize=4, error_kw=dict(lw=1.2), label='HC' if xi==0 else '')
    ax.bar(xi + w/2, nv, width=w, color=C_NPD, alpha=0.75,
           yerr=ne, capsize=4, error_kw=dict(lw=1.2), label='NPD' if xi==0 else '')
    jitter = np.random.default_rng(7).uniform(-0.05, 0.05, N_AGENTS)
    ax.scatter(xi - w/2 + jitter, hd,  c=C_HC,  alpha=0.25, s=10, edgecolors='none')
    ax.scatter(xi + w/2 + jitter, nd, c=C_NPD, alpha=0.25, s=10, edgecolors='none')
    _, p = stats.mannwhitneyu(hd, nd, alternative='two-sided')
    ax.text(xi, max(hv, nv) + 0.04, '***' if p < 0.001 else ('**' if p < 0.01 else '*'),
            ha='center', fontsize=12)

ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel('Mean ± SEM'); ax.set_title('(D)  Group comparison (Mann–Whitney U)')
ax.legend(fontsize=9)

fig2.suptitle('Empathy Dissociation: Cognitive Accuracy Preserved, Affective Concern Impaired in NPD',
              fontsize=12, fontweight='bold', y=1.01)
fig2.savefig('./fig2_empathy_dissociation.pdf', bbox_inches='tight')
fig2.savefig('./fig2_empathy_dissociation.png', bbox_inches='tight', dpi=150)
print("Fig 2 saved.")

# ─────────────────────────────────────────────────────────────────────
# ═══════════  FIGURE 3 – SELF-ESTEEM DYNAMICS  ═══════════════════════
# ─────────────────────────────────────────────────────────────────────
tt_se = np.arange(T + 1)

hc_se_m,  hc_se_se,  hc_se_all  = agg('HC',  'SE')
npd_se_m, npd_se_se, npd_se_all = agg('NPD', 'SE')
hc_pi_m,  hc_pi_se,  _          = agg('HC',  'PI_S')
npd_pi_m, npd_pi_se, _          = agg('NPD', 'PI_S')

fig3 = plt.figure(figsize=(11, 8))
gs   = gridspec.GridSpec(2, 2, figure=fig3, hspace=0.42, wspace=0.35)

# ── 3A: Self-esteem trajectories
ax = fig3.add_subplot(gs[0, :])
ax.axvspan(INJURY_START, INJURY_START + INJURY_LEN, color=C_INJ, alpha=0.28,
           label='Narcissistic injury block')
# Individual agent traces (thin, alpha)
for r in hc_se_all[:10]:
    ax.plot(tt_se, r, color=C_HC,  lw=0.5, alpha=0.18)
for r in npd_se_all[:10]:
    ax.plot(tt_se, r, color=C_NPD, lw=0.5, alpha=0.18)
# Group means ± SEM
ax.fill_between(tt_se,
    gaussian_filter1d(hc_se_m - hc_se_se, 6),
    gaussian_filter1d(hc_se_m + hc_se_se, 6),
    alpha=ALPHA_SHADE + 0.05, color=C_HC)
ax.fill_between(tt_se,
    gaussian_filter1d(npd_se_m - npd_se_se, 6),
    gaussian_filter1d(npd_se_m + npd_se_se, 6),
    alpha=ALPHA_SHADE + 0.05, color=C_NPD)
ax.plot(tt_se, gaussian_filter1d(hc_se_m,  6), color=C_HC,  lw=2.5, label='HC')
ax.plot(tt_se, gaussian_filter1d(npd_se_m, 6), color=C_NPD, lw=2.5, label='NPD')
ax.axhline(0, color='grey', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Trial'); ax.set_ylabel(r'Self-esteem $S(t)$')
ax.set_title('(A)  Self-Esteem Trajectories under Narcissistic Injury Paradigm', fontweight='bold')
ax.legend(fontsize=9)

# ── 3B: Dynamic precision π_self
ax = fig3.add_subplot(gs[1, 0])
ax.axvspan(INJURY_START, INJURY_START + INJURY_LEN, color=C_INJ, alpha=0.25)
ax.fill_between(tt_se,
    gaussian_filter1d(hc_pi_m - hc_pi_se, 6),
    gaussian_filter1d(hc_pi_m + hc_pi_se, 6),
    alpha=ALPHA_SHADE, color=C_HC)
ax.fill_between(tt_se,
    gaussian_filter1d(npd_pi_m - npd_pi_se, 6),
    gaussian_filter1d(npd_pi_m + npd_pi_se, 6),
    alpha=ALPHA_SHADE, color=C_NPD)
ax.plot(tt_se, gaussian_filter1d(hc_pi_m,  6), color=C_HC,  lw=2.0, label='HC')
ax.plot(tt_se, gaussian_filter1d(npd_pi_m, 6), color=C_NPD, lw=2.0, label='NPD')
ax.set_xlabel('Trial'); ax.set_ylabel(r'Self-model precision $\Pi_{self}(t)$')
ax.set_title(r'(B)  Self-Model Precision $\Pi_{self}(t)$ Erosion', fontweight='bold')
ax.legend(fontsize=9)

# ── 3C: Self-esteem variability violin plot
ax = fig3.add_subplot(gs[1, 1])
vp = ax.violinplot([hc_se_var, npd_se_var], positions=[1, 2],
                   showmedians=True, widths=0.5)
for i, (body, col) in enumerate(zip(vp['bodies'], [C_HC, C_NPD])):
    body.set_facecolor(col); body.set_alpha(0.6)
vp['cmedians'].set_color(['white', 'white'])
jit = np.random.default_rng(99).uniform(-0.08, 0.08, N_AGENTS)
ax.scatter(np.ones(N_AGENTS)  + jit, hc_se_var,  c=C_HC,  alpha=0.4, s=12, edgecolors='none')
ax.scatter(np.ones(N_AGENTS)*2 + jit, npd_se_var, c=C_NPD, alpha=0.4, s=12, edgecolors='none')
_, p_var = stats.mannwhitneyu(hc_se_var, npd_se_var, alternative='two-sided')
ax.text(1.5, max(npd_se_var) + 0.02, f'p={p_var:.3e}', ha='center', fontsize=9)
ax.set_xticks([1, 2]); ax.set_xticklabels(['HC', 'NPD'])
ax.set_ylabel(r'$\sigma(S)$ – Self-Esteem Volatility')
ax.set_title('(C)  Self-Esteem Volatility', fontweight='bold')

fig3.suptitle('Self-Esteem Dynamics: Inflated Rigidity and Collapse in NPD',
              fontsize=12, fontweight='bold', y=1.01)
fig3.savefig('./fig3_selfesteem_dynamics.pdf', bbox_inches='tight')
fig3.savefig('./fig3_selfesteem_dynamics.png', bbox_inches='tight', dpi=150)
print("Fig 3 saved.")

# ─────────────────────────────────────────────────────────────────────
# ═══════════  FIGURE 4 – PHASE PORTRAIT  ═════════════════════════════
# ─────────────────────────────────────────────────────────────────────
fig4, axes = plt.subplots(1, 2, figsize=(11, 5.5), gridspec_kw={'wspace': 0.38})

# ── 4A: Phase portrait  (Π_self vs S) for representative agents
ax = axes[0]
for i, r in enumerate(res['HC'][:20]):
    ax.plot(r['SE'][1:], r['PI_S'][1:], color=C_HC, lw=0.7, alpha=0.30)
for i, r in enumerate(res['NPD'][:20]):
    ax.plot(r['SE'][1:], r['PI_S'][1:], color=C_NPD, lw=0.7, alpha=0.30)

# Highlight one representative agent per group
rh = res['HC'][0]
rn = res['NPD'][0]
sc = ax.scatter(rh['SE'][1:], rh['PI_S'][1:],
                c=np.arange(T), cmap='Blues', s=6, alpha=0.7, zorder=4)
sc2 = ax.scatter(rn['SE'][1:], rn['PI_S'][1:],
                 c=np.arange(T), cmap='Reds',  s=6, alpha=0.7, zorder=4)

# Mark start/injury/end
for r, col, lab in [(rh, C_HC, 'HC'), (rn, C_NPD, 'NPD')]:
    ax.scatter(r['SE'][0], r['PI_S'][0], marker='o', s=80, c=col,
               edgecolors='white', linewidths=1.2, zorder=6)
    ax.scatter(r['SE'][INJURY_START], r['PI_S'][INJURY_START], marker='^', s=80, c=col,
               edgecolors='white', linewidths=1.2, zorder=6)
    ax.scatter(r['SE'][INJURY_START + INJURY_LEN], r['PI_S'][INJURY_START + INJURY_LEN],
               marker='s', s=80, c=col, edgecolors='white', linewidths=1.2, zorder=6)

# Grandiose / Vulnerable basin annotations
ax.text( 1.2, 8.5, 'Grandiose basin\n(high $S$, high $\\Pi_{self}$)',
         ha='center', fontsize=9, color=C_NPD,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fef0d0', alpha=0.8))
ax.text(-0.9, 0.5, 'Vulnerable basin\n(low $S$, low $\\Pi_{self}$)',
         ha='center', fontsize=9, color='#8e6bbf',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0e6ff', alpha=0.8))

# Legend handles
legend_handles = [
    mpatches.Patch(color=C_HC,  label='HC (representative)'),
    mpatches.Patch(color=C_NPD, label='NPD (representative)'),
    plt.scatter([], [], marker='o', c='grey', s=60, label='Start'),
    plt.scatter([], [], marker='^', c='grey', s=60, label='Injury onset'),
    plt.scatter([], [], marker='s', c='grey', s=60, label='Injury end'),
]
ax.legend(handles=legend_handles, fontsize=8.5, loc='center right')
ax.set_xlabel(r'Self-esteem $S(t)$')
ax.set_ylabel(r'Self-model precision $\Pi_{self}(t)$')
ax.set_title(r'(A)  Phase Portrait in $(S,\; \Pi_{self})$ Space' + '\n' +
             'NPD: injury drives trajectory from grandiose to vulnerable basin',
             fontweight='bold')

# ── 4B: Potential landscape analogy
ax = axes[1]
# Sketch a double-well potential V(S) for two conditions: normal vs. injury
S_vals = np.linspace(-2.5, 3.0, 500)

# Potential: two wells with adjustable depth ratio
def double_well(S, a1, a2, s1, s2, k):
    """Two Gaussian wells at s1 and s2 with depths a1, a2 and curvature k."""
    return -a1 * np.exp(-k * (S - s1)**2) - a2 * np.exp(-k * (S - s2)**2)

# Normal (pre-injury): deep grandiose well
V_normal = double_well(S_vals, 3.0, 1.0,  1.5, -0.5, 0.8)
# Under injury (pi_self collapsed): shallow grandiose well, deeper vulnerable well
V_injury = double_well(S_vals, 1.2, 2.5,  1.5, -0.8, 0.8)
# HC: balanced wells
V_hc     = double_well(S_vals, 1.8, 1.8,  0.2, -0.2, 1.2)

V_normal -= V_normal.min(); V_normal /= V_normal.max()
V_injury -= V_injury.min(); V_injury /= V_injury.max()
V_hc     -= V_hc.min();     V_hc     /= V_hc.max()

ax.plot(S_vals, V_hc,     color=C_HC,    lw=2.2, label='HC',            zorder=3)
ax.plot(S_vals, V_normal, color=C_NPD,   lw=2.2, label='NPD (normal)',   zorder=3)
ax.plot(S_vals, V_injury, color='#8e6bbf', lw=2.2, ls='--',
        label='NPD (narcissistic injury)', zorder=3)

# Mark minima
for V, col, xs_range in [(V_normal, C_NPD, [0.5, 2.5]), (V_injury, '#8e6bbf', [-1.5, 0.0]),
                          (V_hc, C_HC, [-0.5, 0.5])]:
    mask = (S_vals > xs_range[0]) & (S_vals < xs_range[1])
    if mask.any():
        idx  = np.argmin(V[mask])
        Smin = S_vals[mask][idx]
        Vmin = V[mask][idx]
        ax.scatter(Smin, Vmin, s=80, zorder=6, color=col, edgecolors='white', linewidths=1.2)

# Arrows indicating regime transition
ax.annotate('', xy=(-0.7, 0.35), xytext=(1.5, 0.22),
            arrowprops=dict(arrowstyle='->', color='#8e6bbf', lw=1.5,
                            connectionstyle='arc3,rad=0.35'))
ax.text(0.4, 0.50, 'Precision\ncollapse', ha='center', fontsize=9,
        color='#8e6bbf')

ax.fill_between(S_vals, V_normal, V_injury,
                where=(V_injury < V_normal), alpha=0.12, color='#8e6bbf',
                label='Injury-induced well shift')
ax.set_xlabel(r'Self-esteem $S$')
ax.set_ylabel(r'Free-energy landscape $\mathcal{V}(S)$ (a.u.)')
ax.set_title('(B)  Free-Energy Landscape:\n'
             'Precision Collapse Deepens Vulnerable Well',
             fontweight='bold')
ax.legend(fontsize=8.5, loc='upper left')

fig4.suptitle('Phase Space Analysis: Bistability and Narcissistic Collapse',
              fontsize=12, fontweight='bold', y=1.01)
fig4.savefig('./fig4_phase_portrait.pdf', bbox_inches='tight')
fig4.savefig('./fig4_phase_portrait.png', bbox_inches='tight', dpi=150)
print("Fig 4 saved.")

# ─────────────────────────────────────────────────────────────────────
# ═══════════  FIGURE 5 – PARAMETER SENSITIVITY  ══════════════════════
# ─────────────────────────────────────────────────────────────────────
fig5, axes = plt.subplots(1, 3, figsize=(12, 4.5), gridspec_kw={'wspace': 0.40})

# ── 5A: Effect of varying π_aff on empathic concern
pi_aff_range = np.linspace(0.1, 6.0, 30)
concern_means = []
for paf in pi_aff_range:
    rng_p = np.random.default_rng(100)
    tmp = [simulate_agent(dict(pi_cog=5.0, pi_aff=paf, pi_self=9.0, mu_self_0=1.5),
                          100, 200, 0, np.random.default_rng(rng_p.integers(int(1e6))))
           for _ in range(20)]
    concern_means.append(np.mean([np.mean(r['con']) for r in tmp]))

ax = axes[0]
ax.plot(pi_aff_range, concern_means, color=C_NPD, lw=2.0)
ax.axvline(PARAMS['HC']['pi_aff'],  color=C_HC,  ls='--', lw=1.5,
           label=f'HC  π_aff={PARAMS["HC"]["pi_aff"]}')
ax.axvline(PARAMS['NPD']['pi_aff'], color=C_NPD, ls='--', lw=1.5,
           label=f'NPD π_aff={PARAMS["NPD"]["pi_aff"]}')
ax.set_xlabel(r'Affective precision $\pi_{aff}$')
ax.set_ylabel('Mean Empathic Concern')
ax.set_title(r'(A)  Concern vs. $\pi_{aff}$', fontweight='bold')
ax.legend(fontsize=9)

# ── 5B: Effect of varying π_self on SE volatility
pi_self_range = np.linspace(0.5, 12.0, 25)
se_vol_means = []
for ps in pi_self_range:
    rng_p = np.random.default_rng(200)
    tmp = [simulate_agent(dict(pi_cog=5.0, pi_aff=0.6, pi_self=ps, mu_self_0=1.5),
                          T, INJURY_START, INJURY_LEN,
                          np.random.default_rng(rng_p.integers(int(1e6))))
           for _ in range(20)]
    se_vol_means.append(np.mean([np.std(r['SE']) for r in tmp]))

ax = axes[1]
ax.plot(pi_self_range, se_vol_means, color=C_NPD, lw=2.0)
ax.axvline(PARAMS['HC']['pi_self'],  color=C_HC,  ls='--', lw=1.5,
           label=f'HC  π_self={PARAMS["HC"]["pi_self"]}')
ax.axvline(PARAMS['NPD']['pi_self'], color=C_NPD, ls='--', lw=1.5,
           label=f'NPD π_self={PARAMS["NPD"]["pi_self"]}')
ax.set_xlabel(r'Baseline self-model precision $\pi_{self}$')
ax.set_ylabel(r'Self-Esteem Volatility $\sigma(S)$')
ax.set_title(r'(B)  SE Volatility vs. $\pi_{self}$', fontweight='bold')
ax.legend(fontsize=9)

# ── 5C: Precision ratio π_cog/π_aff vs. empathy imbalance index
ratio_range = np.linspace(0.5, 12, 30)
imbalance   = []
for ratio in ratio_range:
    pi_cog_v = np.sqrt(ratio) * 1.8
    pi_aff_v = 1.8 / np.sqrt(ratio)
    rng_p = np.random.default_rng(300)
    tmp = [simulate_agent(dict(pi_cog=pi_cog_v, pi_aff=pi_aff_v, pi_self=5.0, mu_self_0=0.5),
                          100, 200, 0, np.random.default_rng(rng_p.integers(int(1e6))))
           for _ in range(20)]
    mean_acc = np.mean([np.mean(r['acc']) for r in tmp])
    mean_con = np.mean([np.mean(r['con']) for r in tmp])
    imbalance.append(mean_acc - mean_con)

ax = axes[2]
ax.plot(ratio_range, imbalance, color='#762a83', lw=2.0)
hc_r  = PARAMS['HC']['pi_cog']  / PARAMS['HC']['pi_aff']
npd_r = PARAMS['NPD']['pi_cog'] / PARAMS['NPD']['pi_aff']
ax.axvline(hc_r,  color=C_HC,  ls='--', lw=1.5, label=f'HC  ratio={hc_r:.1f}')
ax.axvline(npd_r, color=C_NPD, ls='--', lw=1.5, label=f'NPD ratio={npd_r:.1f}')
ax.axhline(0, color='grey', lw=0.8, ls=':')
ax.set_xlabel(r'Precision ratio $\pi_{cog}/\pi_{aff}$')
ax.set_ylabel('Empathy Imbalance\n(Accuracy − Concern)')
ax.set_title(r'(C)  Empathy Imbalance vs. $\pi_{cog}/\pi_{aff}$', fontweight='bold')
ax.legend(fontsize=9)

fig5.suptitle('Parameter Sensitivity Analysis',
              fontsize=12, fontweight='bold', y=1.01)
fig5.savefig('./fig5_sensitivity.pdf', bbox_inches='tight')
fig5.savefig('./fig5_sensitivity.png', bbox_inches='tight', dpi=150)
print("Fig 5 saved.")

print("\nAll figures generated successfully.")
